In [19]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-4.1",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [22]:
from langchain.tools import tool  
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import requests
@tool()
def web_search(query: str):
    """Perform a web search about a topic."""
    search = DuckDuckGoSearchResults(num_results=3)
    return (search.invoke(query))
@tool()
def wiki_search(query: str) -> str:
    """Perform a Wikipedia search."""
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return str(wikipedia.run(query))
@tool()
def find_coordinates(city: str) -> str:
    """Find GPS coordinates for a given city, i.e. 'Paris'."""
    api = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&format=json"
    rs = requests.get(api)
    error_msg = "Coordinates not found. Try specifying a proper city."
    if rs.status_code == 200:
        try:
            data = rs.json()
            return f"{data['results'][0]['latitude']}, {data['results'][0]['longitude']}"
        except (KeyError, IndexError):
            return error_msg
    return error_msg



In [23]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field, ValidationError

class HistoricalEvent(BaseModel):
    event_name: str
    exact_date: str = Field(description="ISO-8601 date (YYYY-MM-DD)")
    location: Optional[str]
    coordinates: Optional[str] = Field(
        None,
        description="GPS coordinates in 'latitude, longitude' format."
    )
    event_summary: str
    event_detail: str
    key_figures: List[str]
    # Restricting the link type prevents the model from inventing "creative" stories
    impact_category: Literal["Economic", "Military", "Diplomatic", "Social", "Cultural", "Scientific"]
    groundtruth_sources: Optional[List[str]] = Field(
        None,
        description="List of URLs or references used to compile the information about the event.")
    unverified_assumptions: Optional[str] = Field(
        None,
        description="List of any unverified assumptions made by user or in assistant response.")

historical_event_schema = HistoricalEvent.model_json_schema()

In [24]:
HistoricalEvent.model_json_schema()

{'properties': {'event_name': {'title': 'Event Name', 'type': 'string'},
  'exact_date': {'description': 'ISO-8601 date (YYYY-MM-DD)',
   'title': 'Exact Date',
   'type': 'string'},
  'location': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'title': 'Location'},
  'coordinates': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "GPS coordinates in 'latitude, longitude' format.",
   'title': 'Coordinates'},
  'event_summary': {'title': 'Event Summary', 'type': 'string'},
  'event_detail': {'title': 'Event Detail', 'type': 'string'},
  'key_figures': {'items': {'type': 'string'},
   'title': 'Key Figures',
   'type': 'array'},
  'impact_category': {'enum': ['Economic',
    'Military',
    'Diplomatic',
    'Social',
    'Cultural',
    'Scientific'],
   'title': 'Impact Category',
   'type': 'string'},
  'groundtruth_sources': {'anyOf': [{'items': {'type': 'string'},
     'type': 'array'},
    {'type': 'null'}],
   'default': None,
   'descr

In [25]:

def _generate(model, system_prompt, prompt, tools, format):
    from langchain_core.messages import ToolMessage
    agent = create_agent(
        model,
        system_prompt=system_prompt,
        tools=tools,
        response_format=format
    )
    response = agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]}
    )
    #check tools usage
    tools = set()
    for msg in response["messages"]:
        if isinstance(msg, ToolMessage):
            tools.add(msg.name)
    print(f"Tools used: {tools}")
    try:
        structured_output = response["structured_response"]
        return structured_output
    except (ValidationError, KeyError) as e:
        print("Validation Error:", e)
        return response["messages"][-1].text

In [33]:
# without any output instruction
prompt="""Tell me about the signing of the Treaty of Versailles. 
Provide the exact date, the event location with GPS coordinates, the specific room it happened, and a list of every major head of state present."""
# not explicit forced to use tools
system_prompt="""You are a highly intelligent historical assistant."""

# Often no tools used at all! Unpredictable, and prone to hallucinations
output = _generate(
    "ollama:gpt-oss:20b", #same with gpt-5
    system_prompt,
    prompt,
    [wiki_search, find_coordinates],
    None)
display(output)

Tools used: set()
Validation Error: 'structured_response'


'**Treaty of Versailles – Signing**\n\n| Item | Details |\n|------|---------|\n| **Exact Date** | **28\u202fJune\u202f1919** (the final day of the Paris Peace Conference) |\n| **Event Location** | **Palace of Versailles, Versailles, Île-de-France, France** |\n| **GPS Coordinates** | **48.804722°\u202fN, 2.120833°\u202fE** (approximate center of the palace grounds; the Hall of Mirrors is slightly southwest of this point at 48.805\u202fN,\u202f2.118\u202fE) |\n| **Specific Room** | **Hall of Mirrors** (the “Hall of Mirrors” or “Hall des Miroirs”) – the grand, gilded gallery that served as the ceremonial signing hall. |\n\n---\n\n### Major Heads of State / Government Present at the Signing\n\n| Country | Head of State / Government | Position at Signing |\n|---------|---------------------------|---------------------|\n| **United Kingdom** | **David Lloyd\u202fGeorge** | Prime Minister of the UK |\n| **France** | **Georges\u202fClemenceau** | Premier (Prime Minister) of France |\n| **Italy*

In [36]:
# better tools usage with explicit JSON output requirement (by user)
prompt="""Tell me about the signing of the Treaty of Versailles. 
Provide the exact date, the event location with GPS coordinates, the specific room it happened, and a list of every major head of state present.
Output the response in valid JSON format only, without any additional text, following this schema: 
{
  "event_name": str,
  "exact_date": str,
  "location": str,
  "coordinates": str,
  "event_summary": str,
  "event_detail": str,
  "key_figures": [str],
  "impact_category": str,
  "groundtruth_sources": [str],
  "unverified_assumptions": str
}"""

system_prompt="""You are a highly intelligent historical assistant."""

output = _generate(
    "ollama:gpt-oss:20b",
    system_prompt,
    prompt,
    [wiki_search, find_coordinates],
    None)
display(output)

Tools used: {'find_coordinates', 'wiki_search'}
Validation Error: 'structured_response'


'{\n  "event_name": "Treaty of Versailles Signing",\n  "exact_date": "1919-06-28",\n  "location": "Palace of Versailles, Hall of Mirrors, Versailles, France",\n  "coordinates": "48.80359, 2.13424",\n  "event_summary": "On 28 June 1919, the Treaty of Versailles was signed at the Palace of Versailles, formalizing the peace terms that ended World War I.",\n  "event_detail": "The signing took place in the Hall of Mirrors, a grand chamber of the Palace of Versailles. Delegates from the Allied powers—United States, France, United Kingdom, and Italy—completed the document that imposed reparations, territorial concessions, and the infamous War Guilt Clause on Germany. The treaty marked the official end of hostilities, but its harsh provisions sowed the seeds for future conflict.",\n  "key_figures": [\n    "Woodrow Wilson",\n    "Georges Clemenceau",\n    "David Lloyd George",\n    "Vittorio Emanuele Orlando"\n  ],\n  "impact_category": "Diplomatic",\n  "groundtruth_sources": [\n    "https://en

In [ ]:
# even better with structured output
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy
prompt="""Tell me about the signing of the Treaty of Versailles. 
Provide the exact date, the event location with GPS coordinates, the specific room it happened, and a list of every major head of state present."""

system_prompt="""You are a highly intelligent historical research assistant."""
output = _generate(
    "gpt-4.1",
    system_prompt,
    prompt,
    [wiki_search, find_coordinates],
    ProviderStrategy(HistoricalEvent.model_json_schema())
)
display(output)

Tools used: {'wiki_search', 'find_coordinates'}


{'event_name': 'Signing of the Treaty of Versailles',
 'exact_date': '1919-06-28',
 'location': 'Palace of Versailles, Versailles, France',
 'coordinates': '48.80359, 2.13424',
 'event_summary': 'The Treaty of Versailles was signed at the Palace of Versailles on June 28, 1919, formally ending World War I between Germany and the Allied Powers.',
 'event_detail': "The signing ceremony took place in the Hall of Mirrors (Galerie des Glaces) at the Palace of Versailles. It marked the conclusion of the Paris Peace Conference and imposed significant territorial, military, and economic penalties on Germany. Although the negotiations were conducted by key Allied leaders (often called the 'Big Four'), the official signing was attended by representatives from many countries. The major heads of state present or represented included Georges Clemenceau (France), David Lloyd George (United Kingdom), Vittorio Orlando (Italy), and Woodrow Wilson (USA), along with delegates from many other Allied powers